# 09. Dynamic Array Internals and Python List Traps

A Python `list` is a **dynamic array**: a block of memory holding **references** to objects, which grows when full. Understanding it explains both its speed and its classic bugs.

**Learning order:** HOW A LIST GROWS → AMORTIZED APPEND → COSTS → ALIASING TRAP → SHALLOW VS DEEP COPY → DEFAULT ARGUMENT TRAP → INTERVIEW THINKING

## 1. How a list grows

A list keeps a **capacity** larger than its **length**. `append` writes into the spare space (O(1)). When full, Python allocates a **bigger block** (about 1.125× + a constant in CPython; many textbooks say "doubling"), copies all references, and frees the old block (O(n)). Because resizes are rare, the **average** cost of an append is O(1): **amortized O(1)**.

Elements are references, so a list can hold mixed types, and each slot is 8 bytes (a pointer).

In [1]:
import sys
a, changes = [], []
last = sys.getsizeof(a)
for i in range(64):
    a.append(i)
    size = sys.getsizeof(a)
    if size != last:
        changes.append((len(a), size))              # length at which memory grew
        last = size
print(len(changes) < 15)   # True  (only a handful of reallocations for 64 appends)

True


## 2. Costs recap

- Index, assign, `append`, `pop()`: O(1)
- `insert(i, x)`, `pop(i)`, `remove(x)`, `del lst[i]`: **O(n)** (shift elements)
- `x in lst`, `index`, `count`: O(n)
- Slicing and `copy`: O(k) / O(n)
- Fixed memory trick: if you know the size, `[0] * n` is fast and avoids repeated resizing.

## 3. Trap 1: the aliasing bug in 2D lists

`[[0] * 3] * 3` repeats the **same inner list** three times (three references to one object). Changing one row changes all rows. Build rows with a **comprehension**.

In [2]:
bad = [[0] * 3] * 3
bad[0][0] = 1
print(bad)    # [[1, 0, 0], [1, 0, 0], [1, 0, 0]]   <- all rows changed!

good = [[0] * 3 for _ in range(3)]
good[0][0] = 1
print(good)   # [[1, 0, 0], [0, 0, 0], [0, 0, 0]]
print(bad[0] is bad[1], good[0] is good[1])   # True False

[[1, 0, 0], [1, 0, 0], [1, 0, 0]]
[[1, 0, 0], [0, 0, 0], [0, 0, 0]]
True False


## 4. Trap 2: shallow copy vs deep copy

`b = a` makes **two names for one list**. `a[:]`, `a.copy()`, `list(a)` make a **shallow copy**: a new outer list, but the **same inner objects**. Only `copy.deepcopy` copies everything.

In [3]:
import copy

a = [[1], [2]]
alias = a
shallow = a[:]
deep = copy.deepcopy(a)

shallow[0].append(9)          # inner list is shared
print(a, alias is a, shallow is a)   # [[1, 9], [2]] True False

deep[1].append(7)             # the deep copy is independent
print(a, deep)                # [[1, 9], [2]] [[1], [2, 7]]

[[1, 9], [2]] True False
[[1, 9], [2]] [[1], [2, 7]]


## 5. Trap 3: mutable default argument

Default values are created **once** when the function is defined. A list default is shared by all calls.

In [4]:
def add_item(x, items=[]):                # BUG: the same list every call
    items.append(x)
    return items

print(add_item(1), add_item(2))   # [1, 2] [1, 2]

def add_item_ok(x, items=None):
    if items is None:
        items = []
    items.append(x)
    return items

print(add_item_ok(1), add_item_ok(2))   # [1] [2]

[1, 2] [1, 2]
[1] [2]


## 6. In-place vs new array

Many questions say "**do it in place**" (O(1) extra space). Decide early:
- Modify the input (`nums.sort()`, `nums[i], nums[j] = nums[j], nums[i]`) → O(1) space but destroys the original.
- Return a new list (`sorted(nums)`, comprehension) → O(n) space, safer.

Ask the interviewer if modifying the input is allowed.

## 7. Interview thinking

**Say:** "A Python list is a dynamic array: O(1) indexing and amortized O(1) append, O(n) insert/delete in the middle. I'll build the 2D grid with a comprehension to avoid shared rows."

**Common mistakes:** `[[0]*n]*m`, expecting `b = a[:]` to be a deep copy for nested lists, mutable default arguments, `pop(0)` in a loop, forgetting that slicing copies.

## Quick Revision

- Dynamic array: spare capacity, append amortized O(1), resize copies O(n).
- `[[0]*n]*m` aliases rows. Use `[[0]*n for _ in range(m)]`.
- `a[:]` is shallow. `copy.deepcopy` for nested.
- Mutable default args are shared across calls.
- Ask before modifying the input.